<a href="https://colab.research.google.com/github/RonaldPassos/Spark/blob/main/TesteTratamentoSQL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, lit, trim, regexp_replace, lower, coalesce

spark = SparkSession.builder.appName("TratamentoDeDados").getOrCreate()

dados = [
    ("  Ana Silva  ", "SÃO PAULO", "R$ 5.000,00", None),
    ("joao santos", "rio de janeiro", "R$ 3.200,50", 25),
    (" Maria Oliveira ", "Belo Horizonte", "R$ 12.000,00", 40),
    ("Carlos souza", None, "R$ 1.500,00", 17)
]

colunas = ["nome", "cidade", "salario_bruto", "idade"]

In [14]:
df = spark.createDataFrame(dados, colunas)
df.show()


+----------------+--------------+-------------+-----+
|            nome|        cidade|salario_bruto|idade|
+----------------+--------------+-------------+-----+
|     Ana Silva  |     SÃO PAULO|  R$ 5.000,00| NULL|
|     joao santos|rio de janeiro|  R$ 3.200,50|   25|
| Maria Oliveira |Belo Horizonte| R$ 12.000,00|   40|
|    Carlos souza|          NULL|  R$ 1.500,00|   17|
+----------------+--------------+-------------+-----+



In [16]:
df_tratado = df.withColumn("nome", trim(lower(col("nome")))) \
               .withColumn("cidade", coalesce(col("cidade"), lit("Não Informado"))) \
               .withColumn("cidade", lower(col("cidade"))) \
               .withColumn("salario_limpo", regexp_replace(col("salario_bruto"), r"[R$\s.]", "")) \
               .withColumn("salario_limpo", regexp_replace(col("salario_limpo"), ",", ".").cast("float")) \
               .withColumn("categoria_idade",
                    when(col("idade").isNull(), "Não Identificado")
                   .when(col("idade") >= 18, "Maior de Idade")
                   .otherwise("Menor de Idade")).drop("salario_bruto")

df_tratado.show()

+--------------+--------------+-----+-------------+----------------+
|          nome|        cidade|idade|salario_limpo| categoria_idade|
+--------------+--------------+-----+-------------+----------------+
|     ana silva|     são paulo| NULL|       5000.0|Não Identificado|
|   joao santos|rio de janeiro|   25|       3200.5|  Maior de Idade|
|maria oliveira|belo horizonte|   40|      12000.0|  Maior de Idade|
|  carlos souza| não informado|   17|       1500.0|  Menor de Idade|
+--------------+--------------+-----+-------------+----------------+



In [23]:
from numpy import select
df_tratado.createOrReplaceTempView("tabela_clientes")

df_selecao_sql = spark.sql("""
select nome,
       cidade,
       idade,
       salario_limpo as salario,
       categoria_idade as classificacao_idade
    from
      tabela_clientes
""")

df_selecao_sql.show()

+--------------+--------------+-----+-------+-------------------+
|          nome|        cidade|idade|salario|classificacao_idade|
+--------------+--------------+-----+-------+-------------------+
|     ana silva|     são paulo| NULL| 5000.0|   Não Identificado|
|   joao santos|rio de janeiro|   25| 3200.5|     Maior de Idade|
|maria oliveira|belo horizonte|   40|12000.0|     Maior de Idade|
|  carlos souza| não informado|   17| 1500.0|     Menor de Idade|
+--------------+--------------+-----+-------+-------------------+

